#Housekeeping (Step 1)

In [1]:
import sqlite3

import csv

import urllib.request

import os



# GitHub raw URLs for the three CSV files

# Replace <username> and <repo> with the actual values provided by your instructor

BASE_URL = "https://raw.githubusercontent.com/sbracco2003/Module7/main/"

BOOK_URL   = BASE_URL + "Book.csv"

MEMBER_URL = BASE_URL + "Member.csv"

LOAN_URL   = BASE_URL + "Loan.csv"



# Local paths in the Colab /content directory

BOOK_PATH   = "/content/Book.csv"

MEMBER_PATH = "/content/Member.csv"

LOAN_PATH   = "/content/Loan.csv"



DB_PATH = "/content/library.db"

#Download CSV Files (Step 2)

In [2]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:

    urllib.request.urlretrieve(url, path)

    print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


#Create Database and Tables (Step 3)

In [3]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

# Create Book table
conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo TEXT NOT NULL,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    PRIMARY KEY (callNo)
);
""")

# Create Member table
conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id INTEGER NOT NULL,
    firstname TEXT NOT NULL,
    lastName TEXT NOT NULL,
    PRIMARY KEY (id)
);
""")

# Create Loan table
conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo TEXT NOT NULL,
    id INTEGER NOT NULL,
    dateBorrowed TEXT NOT NULL,
    dateReturned TEXT,
    dateDue TEXT NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created.")

Tables created.


In [4]:
conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()

[('Book',), ('Loan',), ('Member',)]

#Load Book Data (Step 4)

In [5]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',
            (row['callNo'], row['title'], row['author'])
        )

conn.commit()

print("Book rows loaded:",
      conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

Book rows loaded: 11


#Load Member Data (Step 5)

In [6]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()

print("Member rows loaded:",
      conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])

Member rows loaded: 4


#Load Loan Data (Step 6)

In [7]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Convert empty dateReturned to None (→ NULL in SQLite)
        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

        conn.execute(
            '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
               VALUES (?, ?, ?, ?, ?);''',
            (
                row['callNo'],
                int(row['id']),
                row['dateBorrowed'],
                date_returned,
                row['dateDue']
            )
        )

conn.commit()

print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Loan rows loaded: 4


#Query 1 — All Books
Retrieve all columns from the Book table, ordered alphabetically by author last name (hint: SQLite has no last-name function — order by the author column as stored).

In [8]:
query1 = """
SELECT *
FROM Book
ORDER BY author;
"""

rows = conn.execute(query1).fetchall()

for row in rows:
    print(row)

('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


#Query 2 — Books Not Yet Returned
Retrieve the title of each book, and the first and last name of the member who borrowed it, for all loans where dateReturned is NULL (i.e., the book is still out).

This query requires a JOIN across all three tables.

In [9]:
query2 = """
SELECT b.title, m.firstname, m.lastName
FROM Loan l
JOIN Book b ON l.callNo = b.callNo
JOIN Member m ON l.id = m.id
WHERE l.dateReturned IS NULL;
"""

rows = conn.execute(query2).fetchall()

for row in rows:
    print(row)

("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


#Query 3 — Loan History for a Specific Book
Retrieve the full loan history for the book with callNo R 141 E45 2006 — show the member's full name, dateBorrowed, dateDue, and dateReturned. Order by dateBorrowed ascending.

In [10]:
query3 = """
SELECT m.firstname, m.lastName,
       l.dateBorrowed, l.dateDue, l.dateReturned
FROM Loan l
JOIN Member m ON l.id = m.id
WHERE l.callNo = 'R 141 E45 2006'
ORDER BY l.dateBorrowed ASC;
"""

rows = conn.execute(query3).fetchall()

for row in rows:
    print(row)

('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


#Query 4 — Members Who Have Never Borrowed a Book
Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table. Use a LEFT JOIN or a sub-query with NOT IN.

In [11]:
query4 = """
SELECT m.id, m.firstname, m.lastName
FROM Member m
LEFT JOIN Loan l ON m.id = l.id
WHERE l.id IS NULL;
"""

rows = conn.execute(query4).fetchall()

for row in rows:
    print(row)

(4, 'John', 'Martin')


#Query 5 — Count of Loans per Member
Retrieve each member's full name and the total number of loans they have made (including completed ones). Include members with zero loans. Order by number of loans descending.

Hint: Use a LEFT JOIN from Member to Loan combined with COUNT() and GROUP BY.

In [12]:
query5 = """
SELECT m.firstname, m.lastName, COUNT(l.id) AS loan_count
FROM Member m
LEFT JOIN Loan l ON m.id = l.id
GROUP BY m.id, m.firstname, m.lastName
ORDER BY loan_count DESC;
"""

rows = conn.execute(query5).fetchall()

for row in rows:
    print(row)

('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


#Query 6 — Free-Choice Analytical Query
Write one original query of your own design that reveals something interesting or useful about this library dataset. It must use at least one JOIN and at least one aggregate function or WHERE condition not used in Queries 1–5. Precede the query with a Markdown cell that states the business question you are answering and why it is useful.

In [13]:
query6 = """
SELECT m.firstname, m.lastName, COUNT(*) AS books_still_out
FROM Loan l
JOIN Member m ON l.id = m.id
WHERE l.dateReturned IS NULL
GROUP BY m.id, m.firstname, m.lastName
ORDER BY books_still_out DESC;
"""

rows = conn.execute(query6).fetchall()

for row in rows:
    print(row)

('David', 'Martin', 2)


In [14]:
conn.close()

#Summary
One thing I noticed is that the dates are stored as text instead of a proper date format, which could make it harder to do accurate date comparisons or calculations. Also, some fields like dateReturned are blank for books that haven’t been returned yet, so you have to handle those carefully in queries. A limitation of this dataset is that it’s pretty simplified. For example, each book only has one callNo, so it doesn’t account for multiple copies of the same book. In a real library system, you’d expect features like multiple copies, reservations, and fines to be included.